# Universal generation-results comparison

This notebook compares modeled and observed generation for any run produced by `run_9ba_selection.py`: annual, rolling, selected weeks, peak week, monthly, or daily. It reads the standardized daily CSV files in the run directory, so it does not depend on whether the run saved one network, many rolling checkpoints, or no solved network.

The three reporting levels are:

1. **Annual / analyzed-period total** — a complete FY2025–26 run uses the authoritative NITI Dashboard annual benchmark. A partial run is summed only over its populated dates and is never annualized.
2. **Monthly** — only months containing modeled dates are shown. Coverage fields identify partial months.
3. **Daily** — only actual modeled dates are shown, and gaps between selected weeks remain gaps.

The daily source has no separate bio-power or small-hydro chronology, so those observed daily and monthly values remain blank.

In [ ]:
from pathlib import Path

# USER SETTINGS
RESULT_DIR = Path("results/run_results/annual_2025-04-01_2026-03-31_none_uc/LP_annual")
OUTPUT_DIR = None  # None writes below RESULT_DIR/analysis/generation
DAILY_DISPLAY_CARRIER = "coal"

In [ ]:
import sys
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts").exists() and (ROOT.parent / "scripts").exists():
    ROOT = ROOT.parent
if not (ROOT / "scripts").exists():
    raise FileNotFoundError("Start Jupyter in the repository root or notebooks directory")
sys.path.insert(0, str(ROOT / "scripts"))

from results_analysis.generation_comparison import (
    GENERATION_CARRIERS,
    analyze_generation_result,
    export_generation_comparison,
    generation_comparison_figures,
)
from results_analysis import load_analysis_run

result_path = RESULT_DIR if RESULT_DIR.is_absolute() else ROOT / RESULT_DIR
analysis = analyze_generation_result(result_path)
run = load_analysis_run(result_path)
figures = generation_comparison_figures(analysis)
# Suppress automatic inline rendering here; later cells display each figure explicitly.
plt.close("all")

## Run and coverage

In [ ]:
run_summary = pd.Series({
    "result_directory": str(analysis.result_dir),
    "run_mode": analysis.metadata.get("run_mode", analysis.metadata.get("mode", "unknown")),
    "unit_commitment": analysis.metadata.get("unit_commitment", "unknown"),
    "solver_status": analysis.metadata.get("solver_status", "not recorded"),
    "solver_condition": analysis.metadata.get("solver_condition", "not recorded"),
    "first_populated_date": analysis.modeled_daily.index.min().date(),
    "last_populated_date": analysis.modeled_daily.index.max().date(),
    "populated_days": len(analysis.modeled_daily),
    "full_fiscal_year": analysis.full_fiscal_year,
}, name="value")
display(run_summary.to_frame())
display(analysis.coverage.round(3))

## Annual / analyzed-period comparison

For a partial run these totals cover only the populated dates; see the coverage columns.

In [ ]:
comparison_columns = [
    "modeled_gwh",
    "observed_gwh",
    "difference_gwh",
    "difference_pct",
]
remaining_columns = [
    column for column in analysis.period.columns if column not in comparison_columns
]
period_display = analysis.period[comparison_columns + remaining_columns].copy()
total_carriers = [
    carrier
    for carrier in period_display.index
    if carrier not in {"comparable_total", "other_res"}
]
if not period_display.loc[
    ["bio_power", "small_hydro"], ["observed_gwh"]
].notna().all().item():
    total_carriers.append("other_res")
total_row = period_display.loc[total_carriers].sum(numeric_only=True)
total_row["difference_pct"] = (
    100 * total_row["difference_gwh"] / total_row["observed_gwh"]
    if total_row["observed_gwh"] else float("nan")
)
period_display.loc["total_generation"] = total_row
for column in period_display.select_dtypes(include="number"):
    period_display[column] = period_display[column].map(
        lambda value: f"{value:,.0f}" if pd.notna(value) else "—"
    )
display(period_display)
display(figures["period"])

## Monthly comparison

Crosses on the figure mark partial months. Months without modeled dates are omitted.

In [ ]:
display(analysis.monthly.round(3))
display(figures["monthly"])

## Daily comparison

The complete daily table is exported for every carrier. Change `DAILY_DISPLAY_CARRIER` in the settings cell to inspect another carrier in the notebook.

## Imports and exports

Interconnector flows are classified at the Tamil Nadu endpoint. Positive flow into a Tamil Nadu bus is an import; negative flow is an export. AC and DC interconnectors are included, and the tables retain the neighboring system and monthly direction separately.

In [ ]:
# Identify the modeled Tamil Nadu boundary from buses with demand.
tn_buses = set(run.base.loads.bus.dropna().astype(str))
interconnectors = run.base.links.index[run.base.links.carrier.astype(str).isin(["AC", "DC"])]

flow_rows = []
p0 = run.frame("links", "p0").fillna(0.0)
p1 = run.frame("links", "p1").fillna(0.0)
for link in interconnectors:
    bus0 = str(run.base.links.at[link, "bus0"])
    bus1 = str(run.base.links.at[link, "bus1"])
    if bus0 in tn_buses and bus1 not in tn_buses:
        signed_import = -p0[link]
        neighbor = bus1
    elif bus1 in tn_buses and bus0 not in tn_buses:
        signed_import = -p1[link]
        neighbor = bus0
    else:
        continue
    flow_rows.append(pd.DataFrame({
        "neighbor": neighbor,
        "interconnector": link,
        "net_import_mw": signed_import,
    }))

if not flow_rows:
    raise ValueError("No external AC/DC interconnector flows were found")
interconnector_flows = pd.concat(flow_rows).reset_index(names="timestamp")
interconnector_flows["imports_mw"] = interconnector_flows["net_import_mw"].clip(lower=0)
interconnector_flows["exports_mw"] = (-interconnector_flows["net_import_mw"]).clip(lower=0)
interconnector_flows["weight"] = interconnector_flows["timestamp"].map(run.weights).fillna(1.0)
for column in ["imports_mw", "exports_mw", "net_import_mw"]:
    interconnector_flows[column.replace("_mw", "_mwh")] = interconnector_flows[column] * interconnector_flows["weight"]

imports_exports_by_neighbor = (
    interconnector_flows.groupby("neighbor")[["imports_mwh", "exports_mwh", "net_import_mwh"]]
    .sum()
    .div(1000)
    .rename(columns=lambda column: column.replace("mwh", "gwh"))
)
imports_exports_by_neighbor["gross_exchange_gwh"] = imports_exports_by_neighbor.imports_gwh + imports_exports_by_neighbor.exports_gwh
imports_exports_by_neighbor = imports_exports_by_neighbor.sort_values("net_import_gwh", ascending=False)

monthly_imports_exports = (
    interconnector_flows.set_index("timestamp").groupby("neighbor")[["imports_mwh", "exports_mwh"]]
    .resample("MS").sum().div(1000).rename(columns=lambda column: column.replace("mwh", "gwh"))
)
display(imports_exports_by_neighbor.round(3))
display(monthly_imports_exports.round(3))
monthly_imports_exports.unstack("neighbor").plot(kind="bar", figsize=(13, 5), width=0.85)
plt.ylabel("GWh")
plt.title("Monthly modeled imports and exports")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
if DAILY_DISPLAY_CARRIER not in GENERATION_CARRIERS:
    raise ValueError(f"DAILY_DISPLAY_CARRIER must be one of {GENERATION_CARRIERS}")
display(analysis.daily.xs(DAILY_DISPLAY_CARRIER, level="carrier").round(3))
display(figures["daily"])

## Export

Exports the full period, monthly, and daily tables, coverage audit, figures, and a provenance manifest.

In [ ]:
export_path = export_generation_comparison(analysis, OUTPUT_DIR, figures)
imports_exports_by_neighbor.to_csv(export_path / "imports_exports_by_neighbor.csv")
monthly_imports_exports.to_csv(export_path / "monthly_imports_exports.csv")
print(f"Analysis written to: {export_path}")